# COOL AI H4 (me): STEP 1: collect and structure (merge) datasets

### Import libraries 

In [1]:
#Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
#NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Matplotlib is a plotting library for python and pyplot gives us a MatLab like plotting framework. We will use this in our plotter function to plot data.
import matplotlib.pyplot as plt
#Seaborn is a Python data visualization library based on matplotlib. It provides a high-level interface for drawing attractive and informative statistical graphics
import seaborn as sns

## Load and view data 

### Indoor temperature

In [2]:
# Get the database from the DataFoundry link
df_indoor = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/Yy9PQlp3clNidmNyc0pYS1JBV1NlQ1JpbGNKWHBIQlVVMjlwQW9nOFY5UT0=", low_memory=False)
df_indoor = df_indoor[(df_indoor.participant == "H4")] # select the correct participant


# # Clean up the dataframe
df_indoor = df_indoor.drop(["device_id", "sender", "participant", "id", "recipient", "pp1", "pp2", "pp3", "activity", "Unnamed: 7"], axis='columns')
df_indoor = df_indoor.rename(columns={"Temperature": "Temperature_indoor"}, errors="raise")
# df_indoor = df_indoor.rename(columns={"ts": "Timestamp"}, errors="raise")
df_indoor = df_indoor.drop(df_indoor[df_indoor.Temperature_indoor < 0].index) ## Drop temperature values under 0
df_indoor = df_indoor.drop(df_indoor[df_indoor.Temperature_indoor > 40].index) ## Drop temperature values over ...
df_indoor['ts'] = pd.to_datetime(df_indoor['ts']) ## Turn timestamp into datetime dtype
df_indoor['ts'] = df_indoor["ts"].dt.round('min')  ##Round the datestamp column to minutes
df_indoor = df_indoor.resample('10min', on='ts').mean().reset_index() #get one value per 10 minutes
df_indoor = df_indoor.dropna() ## drop rows with empty (NA) cells
df_indoor = df_indoor.drop_duplicates() ## Drop duplicate rowsindex_list= df_indoor2.Timestamp[(df_indoor2.Timestamp >= "2024-08-08 16:00:00") & (df_indoor2.Timestamp <= "2024-08-08 19:20:00")].index.tolist(

## Drop outliers (e.g. rogue readings, tests, when a sensor is moved)
q_low = df_indoor["Temperature_indoor"].quantile(0.01)
q_hi  = df_indoor["Temperature_indoor"].quantile(0.99)
df_indoor = df_indoor[(df_indoor["Temperature_indoor"] < q_hi) & (df_indoor["Temperature_indoor"] > q_low)]

## Get data after 22 august 18:00 since this is when the sensors were installed
# df_indoor = df_indoor[(df_indoor.ts > '2024-09-06 15:00:00')]

display(df_indoor.tail())
display(df_indoor.describe())

,ts,Temperature_indoor
2319,2024-10-04 17:50:00,16.877778
2320,2024-10-04 18:00:00,16.811111
2321,2024-10-04 18:10:00,16.800000
2322,2024-10-04 18:20:00,16.800000
2323,2024-10-04 18:30:00,16.800000


,ts,Temperature_indoor
count,2281,2281.000000
mean,2024-09-26 16:58:27.409031168,19.394342
min,2024-09-18 15:20:00,14.847058
25%,2024-09-22 18:20:00,17.023611
50%,2024-09-26 17:20:00,19.730556
75%,2024-09-30 16:30:00,21.391250
max,2024-10-04 18:30:00,24.732639
std,NaN,2.443368


### Outdoor temperature

In [18]:
# Get the database from the DataFoundry link
df_outdoor = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/b3VLVHc5T0FIeHlDK1FQYVMvcmgxcU4vWk5neXlZZVBiQWQrRjV0OGZVQT0=")

# Clean up the dataframe
df_outdoor = df_outdoor.drop(["sender", "device_id", "sender", "participant", "Unnamed: 7", "id", "recipient", "pp1", "pp2", "pp3", "activity"], axis='columns')
df_outdoor = df_outdoor.rename(columns={"Temperature": "Temperature_outdoor"}, errors="raise")
# df_outdoor = df_outdoor.rename(columns={"sender": "sender_outdoor"}, errors="raise")
df_outdoor = df_outdoor.rename(columns={"ts": "Timestamp"}, errors="raise")
df_outdoor = df_outdoor.drop(df_outdoor[df_outdoor.Temperature_outdoor < 0].index) ## Drop temperature values under 0
df_outdoor = df_outdoor.drop(df_outdoor[df_outdoor.Temperature_outdoor > 35].index) ## Drop temperature values over ..
df_outdoor['Timestamp'] = pd.to_datetime(df_outdoor['Timestamp']) ## Turn timestamp into datetime dtype
df_outdoor['Timestamp'] = df_outdoor["Timestamp"].dt.round('min')  ##Round the datestamp column to minutes
df_outdoor['Temperature_outdoor'] = np.round(df_outdoor['Temperature_outdoor'] * 10) / 10 ## round temperature to 1 decimal
df_outdoor = df_outdoor.dropna() ## drop rows with empty (NA) cells
df_outdoor = df_outdoor.drop_duplicates() ## Drop duplicate rows

## Drop outliers (e.g. rogue readings, tests, when a sensor is moved)
q_low = df_outdoor["Temperature_outdoor"].quantile(0.01)
q_hi  = df_outdoor["Temperature_outdoor"].quantile(0.99)
df_outdoor = df_outdoor[(df_outdoor["Temperature_outdoor"] < q_hi) & (df_outdoor["Temperature_outdoor"] > q_low)]

## Drop rows between 2024-08-08 16:00:00 and 19:20:00 since the sensor was indoors at that time
# df_outdoor = df_outdoor[(df_outdoor.Timestamp < '2024-08-08 21:46:00') | (df_outdoor.Timestamp > '2024-08-08 22:20:00')]
df_outdoor = df_outdoor[(df_outdoor.Timestamp > '2024-09-18 00:00:00') & (df_outdoor.Timestamp < '2024-10-05 00:20:00') ]


# df_outdoor = pd.concat([df_outdoor, df_outdoor1])
display(df_outdoor.tail())
display(df_outdoor.describe())

,Timestamp,Temperature_outdoor
9621,2024-10-04 18:48:00,12.9
9622,2024-10-04 19:47:00,10.9
9623,2024-10-04 20:46:00,9.7
9624,2024-10-04 21:46:00,8.6
9625,2024-10-04 22:45:00,7.8


,Timestamp,Temperature_outdoor
count,386,386.000000
mean,2024-09-26 03:07:13.834196992,14.699223
min,2024-09-18 00:24:00,7.600000
25%,2024-09-21 23:48:45,12.500000
50%,2024-09-25 23:12:00,14.400000
75%,2024-09-30 07:30:00,16.900000
max,2024-10-04 22:45:00,24.100000
std,NaN,3.590655


### Window state

In [21]:
# Get the database from the DataFoundry link
df_window = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/QWM2TVQ4OE9KRWF1UU1tRUxzbXJFS1IxM3hhMVZORnNlSlMvRmVyZUFsdz0=", low_memory=False)
df_window = df_window[(df_window.participant == "H4")] # select the correct participant

# Clean up the dataframe
df_window = df_window.drop(["Unnamed: 7", "device_id", "id", "participant", "recipient", "pp1", "pp2", "pp3", "activity", "light", "participant", "curtain"], axis='columns')

list(df_window)

['ts', 'distance sensor', 'light sensor', 'reed sensor', 'sender', 'window']

In [22]:
# get the left window sensor values
df_window_1 = df_window.loc[(df_window['sender'] == "HHa_window_1")].copy()
df_window_1["curtain_1"] = np.where(df_window_1.loc[:,"distance sensor"] > 35, 1, 0) # set curtain state (0= closed; 1 = open)
df_window_1 = df_window_1.rename(columns={"distance sensor": "distance_1"}, errors="raise")
df_window_1["shade_1"] = np.where(df_window_1.loc[:,"light sensor"] < 2000, 1, 0) # set shade (0= no shade; 1 = yes shade)
df_window_1 = df_window_1.rename(columns={"light sensor": "light_1"}, errors="raise")
df_window_1 = df_window_1.drop(["sender", "window", "reed sensor"], axis='columns') # drop window and reed sensor since this sensor does not measure a door state
df_window_1['ts'] = pd.to_datetime(df_window_1['ts']) #Turn timestamp into datetime dtype
df_window_1['ts'] = df_window_1["ts"].dt.round('min')  ##Round the datestamp column to minutes
df_window_1 = df_window_1.resample('10min', on='ts').last().reset_index() #get one value per 10 minutes
df_window_1  = df_window_1.dropna()
df_window_1 = df_window_1.drop_duplicates() #Drop duplicate rows
# # get the right window sensor values
df_window_2 = df_window.loc[df_window['sender'] == "HHa_window_2"].copy()
df_window_2["curtain_2"] = np.where(df_window_2.loc[:,"distance sensor"] > 35, 1, 0) # set curtain state (0= closed; 1 = open)
df_window_2 = df_window_2.rename(columns={"distance sensor": "distance_2"}, errors="raise")
df_window_2["shade_2"] = np.where(df_window_2.loc[:,"light sensor"] < 2000, 1, 0) # set shade (0= no shade; 1 = yes shade)
df_window_2 = df_window_2.rename(columns={"light sensor": "light_2"}, errors="raise")
df_window_2 = df_window_2.drop(["sender", "window", "reed sensor"], axis='columns') # drop window and reed sensor since this sensor does not measure a door state
df_window_2['ts'] = pd.to_datetime(df_window_2['ts']) #Turn timestamp into datetime dtype
df_window_2['ts'] = df_window_2["ts"].dt.round('min')  ##Round the datestamp column to minutes
df_window_2 = df_window_2.resample('10min', on='ts').last().reset_index() #get one value per 10 minutes
df_window_2  = df_window_2.dropna()
df_window_2 = df_window_2.drop_duplicates() #Drop duplicate rows
# # get the right window sensor values
df_window_3 = df_window.loc[df_window['sender'] == "HHa_window_3"].copy()
df_window_3["curtain_3"] = np.where(df_window_3.loc[:,"distance sensor"] > 35, 1, 0) # set curtain state (0= closed; 1 = open)
df_window_3 = df_window_3.rename(columns={"distance sensor": "distance_3"}, errors="raise")
df_window_3["shade_3"] = np.where(df_window_3.loc[:,"light sensor"] < 2000, 1, 0) # set shade (0= no shade; 1 = yes shade)
df_window_3 = df_window_3.rename(columns={"light sensor": "light_3"}, errors="raise")
df_window_3 = df_window_3.drop(["sender", "window", "reed sensor"], axis='columns') # drop window and reed sensor since this sensor does not measure a door state
df_window_3['ts'] = pd.to_datetime(df_window_3['ts']) #Turn timestamp into datetime dtype
df_window_3['ts'] = df_window_3["ts"].dt.round('min')  ##Round the datestamp column to minutes
df_window_3 = df_window_3.resample('10min', on='ts').last().reset_index() #get one value per 10 minutes
df_window_3  = df_window_3.dropna()
df_window_3 = df_window_3.drop_duplicates() #Drop duplicate rows


# # get the right window sensor values
df_windowdoor_1 = df_window.loc[df_window['sender'] == "HHa_windowdoor_1"].copy()
df_windowdoor_1 = df_windowdoor_1.rename(columns={"reed sensor": "windowdoor_1"}, errors="raise")
df_windowdoor_1 = df_windowdoor_1.drop(["sender", "light sensor", "window", "distance sensor"], axis='columns')
df_windowdoor_1['ts'] = pd.to_datetime(df_windowdoor_1['ts']) #Turn timestamp into datetime dtype
df_windowdoor_1['ts'] = df_windowdoor_1["ts"].dt.round('min')  ##Round the datestamp column to minutes
df_windowdoor_1 = df_windowdoor_1.resample('10min', on='ts').last().reset_index() #get one value per 10 minutes
df_windowdoor_1  = df_windowdoor_1.dropna()
df_windowdoor_1 = df_windowdoor_1.drop_duplicates() #Drop duplicate rows
# # get the right window sensor values
df_windowdoor_2 = df_window.loc[df_window['sender'] == "HHa_windowdoor_2"].copy()
df_windowdoor_2 = df_windowdoor_2.rename(columns={"reed sensor": "windowdoor_2"}, errors="raise")
df_windowdoor_2 = df_windowdoor_2.drop(["sender", "light sensor", "window", "distance sensor"], axis='columns')
df_windowdoor_2['ts'] = pd.to_datetime(df_windowdoor_2['ts']) #Turn timestamp into datetime dtype
df_windowdoor_2['ts'] = df_windowdoor_2["ts"].dt.round('min')  ##Round the datestamp column to minutes
df_windowdoor_2 = df_windowdoor_2.resample('10min', on='ts').last().reset_index() #get one value per 10 minutes
df_windowdoor_2  = df_windowdoor_2.dropna()
df_windowdoor_2 = df_windowdoor_2.drop_duplicates() #Drop duplicate rows



df_window_merge1 = pd.merge_asof(df_window_1.sort_values('ts'), df_window_2.sort_values('ts'), on='ts',tolerance =pd.Timedelta('120 min'))
df_window_merge2 = pd.merge_asof(df_window_3.sort_values('ts'), df_window_merge1.sort_values('ts'), on='ts',tolerance =pd.Timedelta('120 min'))
df_windowdoor_merge1 = pd.merge_asof(df_windowdoor_1.sort_values('ts'), df_windowdoor_2.sort_values('ts'), on='ts',tolerance =pd.Timedelta('120 min'))
df_window = pd.merge_asof(df_window_merge2.sort_values('ts'), df_windowdoor_merge1.sort_values('ts'), on='ts',tolerance =pd.Timedelta('120 min'))
df_window = df_window.drop_duplicates(subset=['ts']) #Drop duplicate rows

display(df_window.tail())
display(df_window.describe())

,ts,distance_3,light_3,curtain_3,shade_3,distance_1,light_1,curtain_1,shade_1,distance_2,light_2,curtain_2,shade_2,windowdoor_1,windowdoor_2
5647,2024-11-25 14:30:00,0.0,48.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5648,2024-11-25 14:40:00,0.0,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5649,2024-11-25 14:50:00,0.0,2.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5650,2024-11-25 15:00:00,0.0,125.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5651,2024-11-25 15:10:00,0.0,132.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,ts,distance_3,light_3,curtain_3,shade_3,distance_1,light_1,curtain_1,shade_1,distance_2,light_2,curtain_2,shade_2,windowdoor_1,windowdoor_2
count,5652,5652.000000,5652.000000,5652.000000,5652.000000,3510.000000,3510.000000,3510.000000,3510.000000,1857.000000,1857.000000,1857.000000,1857.000000,3398.000000,3180.000000
mean,2024-10-09 11:30:42.462845184,305.906405,926.396320,0.931706,0.715145,203.478632,1186.941595,0.519373,0.660399,655.605277,901.525579,0.934841,0.781906,0.334020,0.135535
min,2024-09-18 14:20:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2024-09-29 08:57:30,274.000000,0.000000,1.000000,0.000000,15.000000,0.000000,0.000000,0.000000,726.000000,0.000000,1.000000,1.000000,0.000000,0.000000
50%,2024-10-09 12:25:00,306.000000,0.000000,1.000000,1.000000,190.000000,0.000000,1.000000,1.000000,799.000000,0.000000,1.000000,1.000000,0.000000,0.000000
75%,2024-10-19 08:42:30,333.000000,2298.250000,1.000000,1.000000,291.000000,2556.500000,1.000000,1.000000,800.000000,1788.000000,1.000000,1.000000,1.000000,0.000000
max,2024-11-25 15:10:00,804.000000,3440.000000,1.000000,1.000000,805.000000,4095.000000,1.000000,1.000000,801.000000,3901.000000,1.000000,1.000000,1.000000,1.000000
std,NaN,153.421952,1246.545179,0.252273,0.451385,255.479528,1456.873471,0.499696,0.473641,257.674146,1201.118270,0.246872,0.413063,0.471716,0.342348


### API weather data (outdoor temperature)

In [27]:
# Get the database from the DataFoundry link
df_API = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/N0FsN2loZlVYMWhBaE0rd0l5T2NadzR3YTNBcnovQlJ0SG13dHMxL0U1RT0=")
df_API = df_API[(df_API.participant == "H2")] # select the correct participant

# Clean up the dataframe
df_API = df_API.drop(["id", "participant", "sender", "device_id", "activity", "pp1", "pp2", "pp3", "Unnamed: 7", "recipient"], axis='columns')
df_API['Temperature_API'] = np.round(df_API['Temperature_API'] * 10) / 10 # round temperature to 1 decimal
df_API['Temperature_API_MIN'] = np.round(df_API['Temperature_API_MIN'] * 10) / 10 # round temperature to 1 decimal
df_API['Temperature_API_MAX'] = np.round(df_API['Temperature_API_MAX'] * 10) / 10 # round temperature to 1 decimal
df_API['ts'] = pd.to_datetime(df_API['ts']) # Turn timestamp into datetime dtype
df_API['ts'] = df_API["ts"].dt.round('min')  # Round the datestamp column to minutes
df_API = df_API.resample('10min', on='ts').first().reset_index() #get one value per 10 minutes
df_API = df_API.drop_duplicates() # Drop duplicate rows
df_API = df_API.dropna() # drop rows with empty (NA) cells

## Drop outliers (e.g. rogue readings, tests, when a sensor is moved)
q_low = df_API["Temperature_API"].quantile(0.01)
q_hi  = df_API["Temperature_API"].quantile(0.99)
df_API = df_API[(df_API["Temperature_API"] < q_hi) & (df_API["Temperature_API"] > q_low)]

# Get data after 22 august 18:00 since this is when the sensors were installed
df_API = df_API[(df_API.ts > '2024-09-18 00:00:00') & (df_API.ts < '2024-10-05 00:20:00') ]

display(df_API.tail())
display(df_API.describe())

,ts,Temperature_API,Temperature_API_MAX,Temperature_API_MIN,weather_description,weather_main
5399,2024-10-04 20:30:00,10.1,11.5,8.9,broken clouds,Clouds
5400,2024-10-04 20:40:00,9.1,11.5,7.8,broken clouds,Clouds
5401,2024-10-04 20:50:00,9.0,11.0,7.8,scattered clouds,Clouds
5402,2024-10-04 21:00:00,8.9,10.5,7.8,broken clouds,Clouds
5403,2024-10-04 21:10:00,9.1,9.9,7.1,broken clouds,Clouds


,ts,Temperature_API,Temperature_API_MAX,Temperature_API_MIN
count,2362,2362.000000,2362.000000,2362.000000
mean,2024-09-26 07:53:21.439458304,14.330483,15.417697,13.266596
min,2024-09-18 00:10:00,5.500000,6.600000,3.900000
25%,2024-09-22 03:12:30,12.025000,12.900000,11.000000
50%,2024-09-26 05:55:00,14.100000,14.900000,13.200000
75%,2024-09-30 14:27:30,16.300000,17.200000,15.400000
max,2024-10-04 21:10:00,25.100000,26.600000,23.900000
std,NaN,3.973773,3.982780,3.966118


## MERGE ON TIMESTAMP

In [29]:
# Merge indoor and outdoor temperature datasets first (temperature_indoor vs temperature_outdoor)
df_merge1 = pd.merge_asof(df_indoor.sort_values('ts'), df_API.sort_values('ts'), on='ts',tolerance =pd.Timedelta('120 min'))

# df_merge1 = df_merge1.dropna()
display(df_merge1.tail())
display(df_merge1.describe())

,ts,Temperature_indoor,Temperature_API,Temperature_API_MAX,Temperature_API_MIN,weather_description,weather_main
2276,2024-10-04 17:50:00,16.877778,14.4,15.9,13.8,scattered clouds,Clouds
2277,2024-10-04 18:00:00,16.811111,14.4,15.9,13.8,scattered clouds,Clouds
2278,2024-10-04 18:10:00,16.800000,14.3,15.4,13.8,scattered clouds,Clouds
2279,2024-10-04 18:20:00,16.800000,14.3,15.4,13.8,scattered clouds,Clouds
2280,2024-10-04 18:30:00,16.800000,14.0,15.4,13.2,scattered clouds,Clouds


,ts,Temperature_indoor,Temperature_API,Temperature_API_MAX,Temperature_API_MIN
count,2281,2281.000000,2250.000000,2250.000000,2250.000000
mean,2024-09-26 16:58:27.409031168,19.394342,14.189600,15.281644,13.119378
min,2024-09-18 15:20:00,14.847058,5.500000,6.600000,3.900000
25%,2024-09-22 18:20:00,17.023611,11.900000,12.900000,11.000000
50%,2024-09-26 17:20:00,19.730556,14.100000,14.900000,13.200000
75%,2024-09-30 16:30:00,21.391250,16.300000,17.200000,15.400000
max,2024-10-04 18:30:00,24.732639,24.900000,26.100000,23.700000
std,NaN,2.443368,3.936351,3.915536,3.939881


In [32]:
# merge the first merged dataset with the window state dataset (adds columns light sensor, reed sensor (+window)
df_merge2 = pd.merge_asof(df_merge1.sort_values('ts'), df_window.sort_values('ts'), on='ts',tolerance =pd.Timedelta('120 min'))

display(df_merge2.tail())
display(df_merge2.describe())

,ts,Temperature_indoor,Temperature_API,Temperature_API_MAX,Temperature_API_MIN,weather_description,weather_main,distance_3,light_3,curtain_3,...,distance_1,light_1,curtain_1,shade_1,distance_2,light_2,curtain_2,shade_2,windowdoor_1,windowdoor_2
2276,2024-10-04 17:50:00,16.877778,14.4,15.9,13.8,scattered clouds,Clouds,360.0,2028.0,1.0,...,201.0,2063.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2277,2024-10-04 18:00:00,16.811111,14.4,15.9,13.8,scattered clouds,Clouds,237.0,2045.0,1.0,...,201.0,1868.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
2278,2024-10-04 18:10:00,16.800000,14.3,15.4,13.8,scattered clouds,Clouds,417.0,2165.0,1.0,...,201.0,1743.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
2279,2024-10-04 18:20:00,16.800000,14.3,15.4,13.8,scattered clouds,Clouds,324.0,2061.0,1.0,...,201.0,1690.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
2280,2024-10-04 18:30:00,16.800000,14.0,15.4,13.2,scattered clouds,Clouds,270.0,1839.0,1.0,...,201.0,1526.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN


,ts,Temperature_indoor,Temperature_API,Temperature_API_MAX,Temperature_API_MIN,distance_3,light_3,curtain_3,shade_3,distance_1,light_1,curtain_1,shade_1,distance_2,light_2,curtain_2,shade_2,windowdoor_1,windowdoor_2
count,2281,2281.000000,2250.000000,2250.000000,2250.000000,2170.000000,2170.000000,2170.000000,2170.000000,2153.000000,2153.000000,2153.000000,2153.000000,1759.000000,1759.000000,1759.000000,1759.000000,2006.000000,1769.000000
mean,2024-09-26 16:58:27.409031168,19.394342,14.189600,15.281644,13.119378,301.040553,759.280645,0.838710,0.765899,213.807246,1206.206688,0.626568,0.657222,672.885162,924.309835,0.946561,0.772598,0.390329,0.075749
min,2024-09-18 15:20:00,14.847058,5.500000,6.600000,3.900000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2024-09-22 18:20:00,17.023611,11.900000,12.900000,11.000000,259.000000,0.000000,1.000000,1.000000,16.000000,0.000000,0.000000,0.000000,796.000000,0.000000,1.000000,1.000000,0.000000,0.000000
50%,2024-09-26 17:20:00,19.730556,14.100000,14.900000,13.200000,305.000000,0.000000,1.000000,1.000000,201.000000,0.000000,1.000000,1.000000,799.000000,0.000000,1.000000,1.000000,0.000000,0.000000
75%,2024-09-30 16:30:00,21.391250,16.300000,17.200000,15.400000,336.750000,1894.000000,1.000000,1.000000,307.000000,2559.000000,1.000000,1.000000,800.000000,1849.000000,1.000000,1.000000,1.000000,0.000000
max,2024-10-04 18:30:00,24.732639,24.900000,26.100000,23.700000,804.000000,3440.000000,1.000000,1.000000,805.000000,4095.000000,1.000000,1.000000,801.000000,3901.000000,1.000000,1.000000,1.000000,1.000000
std,NaN,2.443368,3.936351,3.915536,3.939881,195.806543,1165.155671,0.367883,0.423533,221.700496,1454.512998,0.483828,0.474748,244.774790,1191.617998,0.224972,0.419273,0.487946,0.264671


In [39]:
# merge the first merged dataset with the window state dataset (adds columns light sensor, reed sensor (+window)
df_merge3 = pd.merge_asof(df_merge2.sort_values('Timestamp'), df_outdoor.sort_values('Timestamp'), on='Timestamp',tolerance =pd.Timedelta('120 min'))

display(df_merge3.tail())
display(df_merge3.describe())

,Timestamp,Temperature_indoor,Temperature_API,Temperature_API_MAX,Temperature_API_MIN,weather_description,weather_main,distance_3,light_3,curtain_3,...,light_1,curtain_1,shade_1,distance_2,light_2,curtain_2,shade_2,windowdoor_1,windowdoor_2,Temperature_outdoor
2276,2024-10-04 17:50:00,16.877778,14.4,15.9,13.8,scattered clouds,Clouds,360.0,2028.0,1.0,...,2063.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,14.2
2277,2024-10-04 18:00:00,16.811111,14.4,15.9,13.8,scattered clouds,Clouds,237.0,2045.0,1.0,...,1868.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,14.2
2278,2024-10-04 18:10:00,16.800000,14.3,15.4,13.8,scattered clouds,Clouds,417.0,2165.0,1.0,...,1743.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,14.2
2279,2024-10-04 18:20:00,16.800000,14.3,15.4,13.8,scattered clouds,Clouds,324.0,2061.0,1.0,...,1690.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,14.2
2280,2024-10-04 18:30:00,16.800000,14.0,15.4,13.2,scattered clouds,Clouds,270.0,1839.0,1.0,...,1526.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,14.2


,Timestamp,Temperature_indoor,Temperature_API,Temperature_API_MAX,Temperature_API_MIN,distance_3,light_3,curtain_3,shade_3,distance_1,light_1,curtain_1,shade_1,distance_2,light_2,curtain_2,shade_2,windowdoor_1,windowdoor_2,Temperature_outdoor
count,2281,2281.000000,2250.000000,2250.000000,2250.000000,2170.000000,2170.000000,2170.000000,2170.000000,2153.000000,2153.000000,2153.000000,2153.000000,1759.000000,1759.000000,1759.000000,1759.000000,2006.000000,1769.000000,2175.000000
mean,2024-09-26 16:58:27.409031168,19.394342,14.189600,15.281644,13.119378,301.040553,759.280645,0.838710,0.765899,213.807246,1206.206688,0.626568,0.657222,672.885162,924.309835,0.946561,0.772598,0.390329,0.075749,14.581701
min,2024-09-18 15:20:00,14.847058,5.500000,6.600000,3.900000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.600000
25%,2024-09-22 18:20:00,17.023611,11.900000,12.900000,11.000000,259.000000,0.000000,1.000000,1.000000,16.000000,0.000000,0.000000,0.000000,796.000000,0.000000,1.000000,1.000000,0.000000,0.000000,12.300000
50%,2024-09-26 17:20:00,19.730556,14.100000,14.900000,13.200000,305.000000,0.000000,1.000000,1.000000,201.000000,0.000000,1.000000,1.000000,799.000000,0.000000,1.000000,1.000000,0.000000,0.000000,14.400000
75%,2024-09-30 16:30:00,21.391250,16.300000,17.200000,15.400000,336.750000,1894.000000,1.000000,1.000000,307.000000,2559.000000,1.000000,1.000000,800.000000,1849.000000,1.000000,1.000000,1.000000,0.000000,16.900000
max,2024-10-04 18:30:00,24.732639,24.900000,26.100000,23.700000,804.000000,3440.000000,1.000000,1.000000,805.000000,4095.000000,1.000000,1.000000,801.000000,3901.000000,1.000000,1.000000,1.000000,1.000000,23.900000
std,NaN,2.443368,3.936351,3.915536,3.939881,195.806543,1165.155671,0.367883,0.423533,221.700496,1454.512998,0.483828,0.474748,244.774790,1191.617998,0.224972,0.419273,0.487946,0.264671,3.571932


## SAVE AS CSV

In [41]:
df_merge3.to_csv(r"C:\Users\20204113\OneDrive - TU Eindhoven/2_Research/2_CoolAI/3_Jupyter_notebooks/H4/DATA (backups)\1_merged.csv", index = None)